<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Embed documents using watsonx's embedding model**


## Repaired edition for Codespaces
Preparation runs without credentials. Both model sections are opt-in. The IBM section requires your API key, project ID, and service region; `skills-network` is not a portable project ID. Secrets are entered privately or read from environment variables.

Fixed setup, current splitter/embedding imports, checked UTF-8 downloads, three-token truncation, model descriptions, result-variable collisions, and optional exercise configuration. Full model inference was not verified during repair: no IBM credentials were supplied and local model weights were not downloaded.


Estimated time needed: **30** minutes


## Overview


Imagine you work in a company that handles a vast amount of text data, including documents, emails, and reports. Your task is to build an intelligent search system that can quickly and accurately retrieve relevant documents based on user queries. Traditional keyword-based search methods often fail to understand the context and semantics of the queries, leading to poor search results.

To address this challenge, you can use embedding models to convert documents into numerical vectors. These vectors capture the semantic meaning of the text, enabling more accurate and context-aware search capabilities. Document embedding is a powerful technique to convert textual data into numerical vectors, which can then be used for various downstream tasks such as search, classification, clustering, and more.


<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/O_XVADBArH0ck4Wf6uuoBA/embeddings.png" width="60%" alt="embeddings">


In this lab, you will learn how to use embedding models from watsonx.ai and Hugging Face to embed documents. By the end of this lab, you will be able to effectively use these embedding models to transform and utilize textual data in your projects.


## __Table of Contents__

<ol>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-required-ribraries">Installing required libraries</a></li>
            <li><a href="#Load-data">Load data</a></li>
            <li><a href="#Split data">Split data</a></li>
        </ol>
    </li>
    <li>
        <a href="#Watsonx-embedding-model">Watsonx embedding model</a>
        <ol>
            <li><a href="#Model-description">Model description</a></li>
            <li><a href="#Build-model">Build model</a></li>
            <li><a href="#Query-embeddings">Query embeddings</a></li>
            <li><a href="#Document-embeddings">Document embeddings</a></li>
        </ol>
    </li>
    <li>
        <a href="#HuggingFace embedding model">HuggingFace embedding models</a>
        <ol>
            <li><a href="#Model-description">Model description</a></li>
            <li><a href="#Build-model">Build model</a></li>
            <li><a href="#Query-embeddings">Query embeddings</a></li>
            <li><a href="#Document-embeddings">Document embeddings</a></li>
        </ol>
    </li>
</ol>

<a href="#Exercises">Exercises</a>
<ol>
    <li><a href="#Exercise-1---Using-another-watsonx-embedding-model">Exercise 1. Using another watsonx embedding model</a></li>
</ol>


## Objectives

After completing this lab, you will be able to:

 - Prepare and preprocess documents for embedding
 - Use watsonx.ai and Hugging Face embedding models to generate embeddings for your documents


----


## Setup


For this lab, you will use the following libraries:

* [`ibm-watson-ai`](https://ibm.github.io/watsonx-ai-python-sdk/fm_embeddings.html#EmbeddingModels) for using embedding models from IBM's watsonx.ai.
* [`langchain`, `langchain-ibm`, `langchain-community`](https://www.langchain.com/) for using relevant features from LangChain.
* [`sentence-transformers`](https://huggingface.co/sentence-transformers) for using embedding models from HuggingFace.


### Installing required libraries
First choose which sections to run in the next cell. Preparation works without an IBM account or a model download. Set `RUN_WATSONX=True` to call your IBM service, and/or `RUN_HUGGINGFACE=True` to download and run the local model. IBM calls may consume your service allowance; local model download and inference require disk space and memory.

Installation uses your selected notebook interpreter and shows errors. Restart the kernel after installation, then continue below; the saved run options are loaded by the data cell. Re-run setup if you change options. Use a separate environment from older labs.


In [ ]:
import json
import sys
import subprocess
from pathlib import Path
RUN_WATSONX = False
RUN_HUGGINGFACE = False
packages = ['langchain-community==0.4.2', 'langchain-core==1.6.9',
            'langchain-text-splitters==1.1.3', 'requests==2.34.2']
if RUN_WATSONX:
    packages += ['langchain-ibm', 'ibm-watsonx-ai']
if RUN_HUGGINGFACE:
    packages += ['langchain-huggingface', 'sentence-transformers']
print('Installing into:', sys.executable)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', *packages])
Path('embedding_lab_data').mkdir(exist_ok=True)
Path('embedding_lab_data/run_options.json').write_text(json.dumps({
    'watsonx': RUN_WATSONX, 'huggingface': RUN_HUGGINGFACE}), encoding='utf-8')
print('Restart the kernel, then continue below.')


After you install the libraries, restart your kernel. You can do that by clicking the **Restart the kernel** icon.

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/build-a-hotdog-not-hotdog-classifier-guided-project/images/Restarting_the_Kernel.png" width="50%" alt="Restart kernel">


## Load data


A text file has been prepared as the source document for the downstream embedding task.

Now, let's download and load it using LangChain's `TextLoader`.


In [ ]:
from pathlib import Path
import json
import requests
options_path = Path('embedding_lab_data/run_options.json')
options = json.loads(options_path.read_text(encoding='utf-8')) if options_path.exists() else {}
RUN_WATSONX = options.get('watsonx', False)
RUN_HUGGINGFACE = options.get('huggingface', False)
text_path = Path('embedding_lab_data/state-of-the-union.txt')
text_path.parent.mkdir(exist_ok=True)
if not text_path.is_file() or not text_path.stat().st_size:
    response = requests.get('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/i5V3ACEyz6hnYpVq6MTSvg/state-of-the-union.txt', timeout=60)
    response.raise_for_status()
    if not response.content:
        raise ValueError('Empty course document.')
    text_path.write_bytes(response.content)
print('Enabled:', options or 'Preparation only')


In [ ]:
from langchain_community.document_loaders import TextLoader

In [ ]:
loader = TextLoader(str(text_path), encoding="utf-8")
data = loader.load()
assert data and data[0].page_content.strip()

Let's take a look at the document.


In [ ]:
data

## Split data


The model has an input token limit. Here `len` counts **characters**, so the 100-character chunk target is not a token budget. For production, count tokens using the chosen model's tokenizer. The small chunks here keep this exercise inexpensive, though larger coherent passages may retrieve better.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
)

In [ ]:
chunks = text_splitter.split_text(data[0].page_content)

Let's see how many chunks you get.


In [ ]:
len(chunks)

Let's also see what these chunks looks like.


In [ ]:
chunks

## Watsonx embedding model


### Model description


The original course used `ibm/slate-125m-english-rtrvr-v2`. This repaired copy defaults to the Granite model in the current [WatsonxEmbeddings integration example](https://docs.langchain.com/oss/python/integrations/embeddings/ibm_watsonx). Set `WATSONX_EMBEDDING_MODEL_ID` to a model available in your service region/project. Model availability, input limits, and vector dimensions are model-specific.


<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/NDCHhZfcC96jggb2hMdJhg/fm-slate-125m-english-rtrvr-cosine.jpg" width="50%">


The original Slate model produced 768-dimensional vectors. Do not assume that dimension for another model; inspect the actual response below.


Consult your selected model's IBM model card for input limits and embedding dimension. Service access must be enabled for your project and region.


### Build model


The following code shows how to build the `slate-125m-english-rtrvr-v2` model from IBM watsonx.ai API.


First, import the necessary dependencies. 
- `WatsonxEmbeddings` is a class/dependence that can be used to form an embedding model object.
- `EmbedTextParamsMetaNames` is a dependence that controls the embedding parameters.


In [ ]:
import os
from getpass import getpass
watsonx_embedding = None
if RUN_WATSONX:
    from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames
    from langchain_ibm import WatsonxEmbeddings


In [ ]:
if RUN_WATSONX:
    api_key = os.environ.get('WATSONX_APIKEY') or getpass('IBM Cloud API key (hidden): ')
    project_id = os.environ.get('WATSONX_PROJECT_ID') or input('Your watsonx project ID: ').strip()
    service_url = os.environ.get('WATSONX_URL') or input('Service URL [https://us-south.ml.cloud.ibm.com]: ').strip() or 'https://us-south.ml.cloud.ibm.com'
    model_id = os.environ.get('WATSONX_EMBEDDING_MODEL_ID', 'ibm/granite-embedding-107m-multilingual')
    if not api_key or not project_id or project_id == 'skills-network':
        raise ValueError('Provide your own IBM Cloud API key and watsonx project ID.')
    # Use the model default limit instead of discarding all but three tokens.
    watsonx_embedding = WatsonxEmbeddings(model_id=model_id, url=service_url,
        project_id=project_id, apikey=api_key,
        params={EmbedTextParamsMetaNames.RETURN_OPTIONS: {'input_text': True}})
    del api_key
else:
    print('Watsonx skipped. Enable RUN_WATSONX in setup and supply your own account details.')


### Query embeddings


Now, create an embedding based on a single sentence, which can be treated as a query.


Use the `embed_query` method.


In [ ]:
query = 'How are you?'
watsonx_query_result = watsonx_embedding.embed_query(query) if watsonx_embedding else None


Let's see the length/dimension of this embedding.


In [ ]:
print("Watsonx dimension:", len(watsonx_query_result) if watsonx_query_result is not None else "not run")

The printed dimension belongs to the selected model. Query and document vectors must come from the same model before they can be compared.


Next, take a look at the first five results from the embeddings.


In [ ]:
print(watsonx_query_result[:5] if watsonx_query_result is not None else "Watsonx not run")

### Document embeddings


After creating the query embeddings, you will be guided on how to create embeddings from documents, which are a list a text chunks.


Use `embed_documents`. The parameter `chunks` should be a list of text. Here, chunks is a list of documents you get from before after splitting the whole document.


In [ ]:
# Start with five chunks; enable the whole document explicitly if needed.
EMBED_ALL_CHUNKS = False
selected_chunks = chunks if EMBED_ALL_CHUNKS else chunks[:5]
watsonx_doc_result = watsonx_embedding.embed_documents(selected_chunks) if watsonx_embedding else None
if watsonx_doc_result is not None:
    assert len(watsonx_doc_result) == len(selected_chunks)
    assert all(len(v) == len(watsonx_query_result) for v in watsonx_doc_result)


One vector is returned per submitted chunk. By default this lesson embeds only the first five chunks; use `EMBED_ALL_CHUNKS=True` above for the full document.


In [ ]:
print(len(watsonx_doc_result) if watsonx_doc_result is not None else "Watsonx not run")

Now, take a look at the first five results from the embeddings of the first piece of text.


In [ ]:
print(watsonx_doc_result[0][:5] if watsonx_doc_result else "Watsonx not run")

Inspect the dimension of the document vector; it should match the query vector from this same model.


In [ ]:
print(len(watsonx_doc_result[0]) if watsonx_doc_result else "Watsonx not run")

## Hugging Face embedding model


### Model description


The local example uses `sentence-transformers/all-mpnet-base-v2`, a 768-dimensional sentence embedding model based on MPNet (not “money-base”). See its [model card](https://huggingface.co/sentence-transformers/all-mpnet-base-v2). This local model needs no IBM credentials but downloads model weights the first time it runs.


### Build model


To build the model, you need to import the `HuggingFaceEmbeddings` dependence first.


In [ ]:
huggingface_embedding = None
if RUN_HUGGINGFACE:
    from langchain_huggingface import HuggingFaceEmbeddings


Then, you specify the model name.


In [ ]:
model_name = "sentence-transformers/all-mpnet-base-v2"

Here we create a embedding model object.


In [ ]:
if RUN_HUGGINGFACE:
    huggingface_embedding = HuggingFaceEmbeddings(model_name=model_name,
        model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True})
else:
    print('Local model skipped. Enable RUN_HUGGINGFACE in setup to download and run it.')


### Query embeddings


Let's create the embeddings from the same sentence, but using the Hugging Face embedding model. 


In [ ]:
query = "How are you?"

In [ ]:
hf_query_result = huggingface_embedding.embed_query(query) if huggingface_embedding else None

In [ ]:
print(hf_query_result[:5] if hf_query_result is not None else "Local model not run")

Raw coordinates from different embedding models cannot be compared directly, even when their vector dimensions match. Compare retrieval performance on the same questions instead.


### Document embeddings


Next, you can do the same for creating embeddings from documents.


In [ ]:
hf_doc_result = huggingface_embedding.embed_documents(selected_chunks) if huggingface_embedding else None
if hf_doc_result is not None:
    assert len(hf_doc_result) == len(selected_chunks)
    assert all(len(v) == len(hf_query_result) for v in hf_doc_result)
    print(hf_doc_result[0][:5])
else:
    print('Local model not run')


In [ ]:
print(len(hf_doc_result[0]) if hf_doc_result else "Local model not run")

# Exercises


### Exercise 1 - Using another watsonx embedding model
Watsonx.ai also supports other embedding models, for which you can find more information [here](https://www.ibm.com/products/watsonx-ai/foundation-models?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-Embed+documents+with+watsonx%E2%80%99s+embedding-v1_1761934835#embedding). Can you try to use another embedding model to create embeddings for the document?


In [ ]:
# Choose another supported embedding model from your watsonx catalog.
# Set its ID explicitly; leaving this blank makes the exercise optional.
SECOND_WATSONX_MODEL_ID = ''
if RUN_WATSONX and SECOND_WATSONX_MODEL_ID:
    from copy import copy
    # Reuse the configured credentials; supply a different model at construction.
    second_model = WatsonxEmbeddings(model_id=SECOND_WATSONX_MODEL_ID,
        url=service_url, project_id=project_id,
        apikey=os.environ.get('WATSONX_APIKEY') or getpass('IBM API key (hidden): '))
    second_vectors = second_model.embed_documents(selected_chunks)
    print('Vectors:', len(second_vectors), 'dimension:', len(second_vectors[0]))
else:
    print('Optional second-model exercise skipped.')


This exercise requires an available model ID from your own IBM account. No hard-coded course project or unsupported model is substituted.


In [ ]:
assert chunks and all(0 < len(c) <= 100 for c in chunks)
assert all(isinstance(c, str) for c in chunks)
print('PASS: document loaded and chunks validated.')
print('Watsonx executed:', watsonx_doc_result is not None)
print('Hugging Face executed:', hf_doc_result is not None)


## Authors


[Kang Wang](https://author.skills.network/instructors/kang_wang)

Kang Wang is a Data Scientist in IBM. He is also a PhD Candidate in the University of Waterloo.


### Other Contributors


[Joseph Santarcangelo](https://author.skills.network/instructors/joseph_santarcangelo)

Joseph has a Ph.D. in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD.

[Hailey Quach](https://author.skills.network/instructors/hailey_quach)

Hailey is a Data Scientist Intern at IBM. She is also pursuing a BSc in Computer Science, Honors at Concordia University, Montreal.


```{## Change Log}
```


```{|Date (YYYY-MM-DD)|Version|Changed By|Change Description||-|-|-|-||2024-07-22|0.1|Kang Wang|Create the lab|}
```


Copyright © IBM Corporation. All rights reserved.
